# TL-Bot - char_classifier Training

**Before the very first run:**
- **Colab:** set runtime to GPU -- *Runtime -> Change runtime type -> T4 GPU -> Save*
- **Lightning AI:** open a Studio with a T4 GPU before running this notebook
- **Kaggle:** enable GPU (*Settings → Accelerator → GPU T4 x2*); add `RCLONE_CONFIG` secret; upload `char-dataset.zip` to Drive (Cell 2 pulls it automatically — no Kaggle dataset upload needed)
- **Local:** ensure `.venv` is active and a CUDA GPU is available (CPU works for smoke tests)

**Every session: set `PLATFORM` in Cell 1, then click Runtime -> Run all.**
- Cell 1 sets the platform, scripts, and epoch count.
- Cell 2 mounts Drive (Colab), confirms persistent storage (Lightning), syncs checkpoints and dataset from Drive via rclone (Kaggle), or confirms local paths.
- Cell 3 clones or pulls the repo then starts training. Resumes automatically.
- Cell 4 (Kaggle only) syncs checkpoints back to Drive after training.

Checkpoints are saved after every epoch and persist across sessions on all platforms.

---
**One-time prerequisites:**

```powershell
# Zip the dataset and upload to your storage root (Colab / Lightning / Kaggle)
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Colab:     upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/
# Lightning: upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
# Kaggle:    upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/ (same as Colab — rclone pulls it in Cell 2)
# Local:     dataset is already present -- no zip needed
```

**Kaggle — rclone setup (run once locally):**
```bash
rclone config
# → New remote → name: gdrive → type: Google Drive → follow OAuth flow
# Copy the full contents of %APPDATA%\rclone\rclone.conf (Windows)
# Paste it into a Kaggle Secret named RCLONE_CONFIG (notebook → Add-ons → Secrets)
```

---
## Cell 1 - Configure
Set the platform, scripts, and epoch count for this run. Edit here only.

In [ ]:
def _detect_platform():
    import os
    # Check Kaggle before Colab — Kaggle also ships the google.colab package
    if os.path.isdir("/kaggle/input"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.isdir("/teamspace/studios/this_studio"):
        return "lightning"
    return "local"

# Platform: "colab" | "lightning" | "kaggle" | "local"
# Auto-detected from the runtime environment.
# Override by assigning explicitly, e.g.: PLATFORM = "colab"
PLATFORM = _detect_platform()
print(f"Platform: {PLATFORM!r}")

# Scripts to train. Options: "latin" | "kana" | "hangul" | "cjk" | "all"
# Single script  -> checkpoint saved to checkpoints/<script>/
# "all"          -> checkpoint saved to checkpoints/  (flat, combined model)
SCRIPTS = ["latin"] #, "kana", "hangul", "cjk"]

# Number of training epochs.
# Recommended: latin=60, kana=60, hangul=60, cjk=60, all=80
EPOCHS = 60

# --- Platform-specific paths (edit the one that matches your PLATFORM) ---

# Colab: Google Drive folder for checkpoints and dataset zip.
COLAB_ROOT = "/content/drive/MyDrive/Colab Notebooks/TL-Bot"

# Lightning AI: persistent studio storage path.
LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

# Kaggle: rclone syncs checkpoints and char-dataset.zip to/from Google Drive
# (same Drive folder as Colab — no separate Kaggle dataset upload needed).
# Prerequisites:
#   1. Run `rclone config` locally (remote name: gdrive, type: Google Drive), then add
#      the full %APPDATA%\rclone\rclone.conf as a Kaggle Secret named RCLONE_CONFIG.
#   2. Upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/ (same zip as Colab).
KAGGLE_ROOT = "/kaggle/working/TL-Bot"

# Local: repo root (empty = use current working directory) and checkpoint output dir.
LOCAL_REPO = ""  # e.g. r"C:\Users\you\Documents\Discord-TL_Bot"
LOCAL_ROOT = str(__import__("pathlib").Path.home() / "tl-bot-checkpoints")

---
## Cell 2 - Setup
Mounts Drive (Colab), confirms persistent storage (Lightning), or confirms local paths.

In [ ]:
import os
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
elif PLATFORM == "lightning":
    os.makedirs(LIGHTNING_ROOT, exist_ok=True)
    print(f"Storage ready: {LIGHTNING_ROOT}")
elif PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient
    from pathlib import Path
    import subprocess
    import zipfile as _zipfile

    # Install rclone
    print("Installing rclone ...")
    subprocess.run(
        "curl -fsSL https://rclone.org/install.sh | sudo bash",
        shell=True, check=True,
    )
    print("rclone installed.")

    # Write rclone config from Kaggle Secret
    cfg = Path.home() / ".config" / "rclone" / "rclone.conf"
    cfg.parent.mkdir(parents=True, exist_ok=True)
    cfg.write_text(UserSecretsClient().get_secret("RCLONE_CONFIG"))
    print("rclone config written.")

    # Pull checkpoints from Drive (same folder as Colab; empty on first run is fine)
    ckpt_dst = Path(KAGGLE_ROOT) / "checkpoints"
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    print("Syncing checkpoints from Drive ...")
    result = subprocess.run([
        "rclone", "sync",
        "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
        str(ckpt_dst), "--progress",
    ])
    if result.returncode != 0:
        print("Warning: rclone sync returned non-zero — check RCLONE_CONFIG secret and gdrive remote name.")
    else:
        print("Checkpoint sync complete.")

    # Pull dataset from Drive (skip if already extracted this session)
    ds_dir = Path(KAGGLE_ROOT) / "char-dataset"
    if not ds_dir.exists():
        ds_zip = Path(KAGGLE_ROOT) / "char-dataset.zip"
        print("Pulling char-dataset.zip from Drive ...")
        subprocess.run([
            "rclone", "copy",
            "gdrive:Colab Notebooks/TL-Bot/char-dataset.zip",
            str(Path(KAGGLE_ROOT)),
            "--progress",
        ], check=True)
        print("Extracting ...")
        with _zipfile.ZipFile(ds_zip, "r") as zf:
            zf.extractall(Path(KAGGLE_ROOT))
        ds_zip.unlink()
        print(f"Dataset ready: {ds_dir}")
    else:
        print(f"Dataset already present: {ds_dir}")
elif PLATFORM == "local":
    from pathlib import Path
    _repo = LOCAL_REPO or os.getcwd()
    _ckpt = LOCAL_ROOT
    Path(_ckpt).mkdir(parents=True, exist_ok=True)
    print(f"Repo  : {_repo}")
    print(f"Ckpts : {_ckpt}")
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}  -- use 'colab', 'lightning', 'kaggle', or 'local'")

---
## Cell 3 - Train
Clones or pulls the repo (Colab/Lightning), then starts or resumes training.

In [ ]:
import os, subprocess, json as _json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"

if PLATFORM == "colab":
    REPO_DIR     = "/content/Discord-TL_Bot"
    STORAGE_ROOT = COLAB_ROOT
    storage_args = ["--storage-root", COLAB_ROOT]
elif PLATFORM == "lightning":
    REPO_DIR     = f"{LIGHTNING_ROOT}/Discord-TL_Bot"
    STORAGE_ROOT = LIGHTNING_ROOT
    storage_args = ["--storage-root", LIGHTNING_ROOT]
elif PLATFORM == "kaggle":
    REPO_DIR     = "/kaggle/working/Discord-TL_Bot"
    STORAGE_ROOT = KAGGLE_ROOT
    storage_args = ["--storage-root", KAGGLE_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR,
                    "--sync-to", "gdrive:Colab Notebooks/TL-Bot/checkpoints/"]
elif PLATFORM == "local":
    REPO_DIR     = LOCAL_REPO or os.getcwd()
    STORAGE_ROOT = LOCAL_ROOT
    storage_args = ["--storage-root", LOCAL_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR]
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

# --- Progress check: read progress.json before starting ---
def _ckpt_dir(root, scripts):
    _all = {"latin", "kana", "hangul", "cjk"}
    s = _all if "all" in scripts else set(scripts)
    if s >= _all:
        return _Path(root) / "checkpoints"
    if len(scripts) == 1:
        return _Path(root) / "checkpoints" / scripts[0]
    return _Path(root) / "checkpoints" / "_".join(sorted(s))

_progress = _ckpt_dir(STORAGE_ROOT, SCRIPTS) / "progress.json"
if _progress.exists():
    p = _json.loads(_progress.read_text())
    print(f"[resume] {_progress}")
    for k, v in p.items():
        if k == "history":
            continue
        print(f"  {k:<22}: {v}")
else:
    print(f"[resume] No progress.json at {_progress} — will start from scratch.")

# Clone / pull for remote platforms; local repo is already present.
if PLATFORM in ("colab", "lightning", "kaggle"):
    os.makedirs(REPO_DIR, exist_ok=True)
    if os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Kaggle: symlink the dataset (pulled from Drive in Cell 2) into the repo tree so train.py finds it.
if PLATFORM == "kaggle":
    _ds_src = str(_Path(KAGGLE_ROOT) / "char-dataset")
    _ds_dst = f"{REPO_DIR}/Models/Datasets/char-dataset"
    if not os.path.exists(_ds_dst):
        os.symlink(_ds_src, _ds_dst)
        print(f"Dataset linked: {_ds_src} → {_ds_dst}")

subprocess.run(
    [
        "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
        "--skip-clone", "--resume",
        "--scripts", *SCRIPTS,
        "--epochs", str(EPOCHS),
        *storage_args,
    ],
    check=True,
)

---
## Cell 4 - Sync (Kaggle emergency fallback)
On Kaggle, checkpoints are automatically synced to Drive every 10 minutes during training — **you do not need to run this cell under normal conditions.**

Run it manually only if the session crashed before the next auto-sync and you want to push whatever checkpoints exist right now.

In [ ]:
if PLATFORM == "kaggle":
    import subprocess
    from pathlib import Path
    ckpt_src = Path(KAGGLE_ROOT) / "checkpoints"
    print(f"Syncing {ckpt_src} → Drive ...")
    subprocess.run([
        "rclone", "sync",
        str(ckpt_src),
        "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
        "--progress",
    ], check=True)
    print("Done — checkpoints saved to Drive.")
else:
    print(f"Platform is {PLATFORM!r} — no sync needed (Drive is live-mounted or local).")